In [1]:
%pip install Flask-SocketIO 
%pip install Flask
%pip install Flask-Cors

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from flask import Flask, request
from flask_socketio import SocketIO, emit, join_room, leave_room

app = Flask(__name__)
app.secret_key = 'random secret key!'
socketio = SocketIO(app, cors_allowed_origins="*")

rooms = {}

@socketio.on('hello')
def hello(message):
    emit('world')

@socketio.on('howdy')
def howdy(message):
    print("Client Message:" , message)


@socketio.on("offer")
def handle_offer(data):
    """Handles sending an offer to the other peer."""
    room_id = data["room"]
    emit("offer", data, room=room_id, include_self=False)
    print("Recieved offer")

@socketio.on("answer")
def handle_answer(data):
    """Handles sending an answer to the other peer."""
    room_id = data["room"]
    emit("answer", data, room=room_id, include_self=False)

@socketio.on("candidate")
def handle_candidate(data):
    """Handles sending ICE candidates to the other peer."""
    room_id = data["room"]
    emit("candidate", data, room=room_id, include_self=False)

@socketio.on("join")
def handle_join(data):
    print("""Handles when a client joins a call.""")
    room_id = data["room"]
    join_room(room_id)

    if room_id not in rooms:
        rooms[room_id] = []

    rooms[room_id].append(request.sid)
    print(f"Client {request.sid} joined room {room_id}")

    if len(rooms[room_id]) > 1:
        emit("ready", {}, room=room_id)

@socketio.on("leave")
def handle_leave(data):
    """Handles when a client leaves a call."""
    room_id = data["room"]
    leave_room(room_id)
    
    if room_id in rooms and request.sid in rooms[room_id]:
        rooms[room_id].remove(request.sid)
        if not rooms[room_id]:
            del rooms[room_id]
        else:
            # Notify remaining user that the other peer has left
            emit("peer_left", {}, room=room_id)

    print(f"Client {request.sid} left room {room_id}")                  


socketio.run(app,  port=5001, allow_unsafe_werkzeug=True )

Handles when a client joins a call.
Client vTjMa-8UC41brwFVAAAU joined room random-room
Recieved offer
Handles when a client joins a call.
Client SaPDwJPzZmzMsvXJAAAN joined room random-room
Recieved offer
Handles when a client joins a call.
Client Z-lNAQbtd20R7uF5AAAr joined room random-room
Recieved offer
Handles when a client joins a call.
Client Z-lNAQbtd20R7uF5AAAr joined room random-room
Recieved offer
Handles when a client joins a call.
Client eON2sbOloTcJ3mACAABC joined room random-room
Recieved offer
Handles when a client joins a call.
Client FsirwMAA6XWclDpFAABf joined room random-room
Recieved offer
Handles when a client joins a call.
Client CD-8CmLVetaByntWAABu joined room random-room
Recieved offer
